In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from sklearn.utils import resample
from sklearn.linear_model import LassoCV, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# 🎯 Lasso Regression on Investment Behavior Data

## What Are We Predicting?

**Target Variable:** `Mutual_Funds`
A score (1–5) representing how strongly an individual leans toward investing in Mutual Funds.

We are answering the question:

> *"Given a person's age, gender, investment goals, risk appetite, and behavior - how much will they lean toward Mutual Funds?"*

Although we focus on `Mutual_Funds`, this model can be retargeted to predict any of the other investment scores by changing a single line in the wrangle function:

| Target | Question Answered |
|---|---|
| `Mutual_Funds` | How strongly does this person lean toward Mutual Funds? |
| `Equity_Market` | How likely are they to invest in the stock market? |
| `Gold` | How much do they prefer Gold as an investment? |
| `Fixed_Deposits` | How strongly do they favor Fixed Deposits? |
| `Government_Bonds` | How likely are they to invest in Government Bonds? |
| `Debentures` | How much do they lean toward Debentures? |
| `PPF` | How strongly do they favor the Public Provident Fund? |

In [31]:
finance = pd.read_csv("/workspaces/Machine-Learining-Models/day_004_lasso_regression/finance/Finance_data.csv")

In [32]:
finance.head()

,gender,age,Investment_Avenues,Mutual_Funds,Equity_Market,Debentures,Government_Bonds,Fixed_Deposits,PPF,Gold,Stock_Marktet,Factor,Objective,Purpose,Duration,Invest_Monitor,Expect,Avenue,What are your savings objectives?,Reason_Equity,Reason_Mutual,Reason_Bonds,Reason_FD,Source
0,Female,34,Yes,1,2,5,3,7,6,4,Yes,Returns,Capital Appreciation,Wealth Creation,1-3 years,Monthly,20%-30%,Mutual Fund,Retirement Plan,Capital Appreciation,Better Returns,Safe Investment,Fixed Returns,Newspapers and Magazines
1,Female,23,Yes,4,3,2,1,5,6,7,No,Locking Period,Capital Appreciation,Wealth Creation,More than 5 years,Weekly,20%-30%,Mutual Fund,Health Care,Dividend,Better Returns,Safe Investment,High Interest Rates,Financial Consultants
2,Male,30,Yes,3,6,4,2,5,1,7,Yes,Returns,Capital Appreciation,Wealth Creation,3-5 years,Daily,20%-30%,Equity,Retirement Plan,Capital Appreciation,Tax Benefits,Assured Returns,Fixed Returns,Television
3,Male,22,Yes,2,1,3,7,6,4,5,Yes,Returns,Income,Wealth Creation,Less than 1 year,Daily,10%-20%,Equity,Retirement Plan,Dividend,Fund Diversification,Tax Incentives,High Interest Rates,Internet
4,Female,24,No,2,1,3,6,4,5,7,No,Returns,Income,Wealth Creation,Less than 1 year,Daily,20%-30%,Equity,Retirement Plan,Capital Appreciation,Better Returns,Safe Investment,Risk Free,Internet


In [33]:
finance.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   gender                             40 non-null     str  
 1   age                                40 non-null     int64
 2   Investment_Avenues                 40 non-null     str  
 3   Mutual_Funds                       40 non-null     int64
 4   Equity_Market                      40 non-null     int64
 5   Debentures                         40 non-null     int64
 6   Government_Bonds                   40 non-null     int64
 7   Fixed_Deposits                     40 non-null     int64
 8   PPF                                40 non-null     int64
 9   Gold                               40 non-null     int64
 10  Stock_Marktet                      40 non-null     str  
 11  Factor                             40 non-null     str  
 12  Objective                          

# Wrangle and Split

In [66]:

def wrangle(filepath):
    # Load raw data
    df = pd.read_csv(filepath)
    
    #  Ordinal encoding (on original strings, before split)
    duration_order = {
        "Less than 1 year": 1,
        "1-3 years": 2,
        "3-5 years": 3,
        "More than 5 years": 4
    }
    expect_order = {
        "10%-20%": 1,
        "20%-30%": 2,
        "30%-40%": 3
    }
    df["Duration"] = df["Duration"].map(duration_order)
    df["Expect"] = df["Expect"].map(expect_order)
    
    #  Drop unnecessary columns
    drop_cols = [
        "Investment_Avenues",
        "Avenue",
        "Stock_Marktet",
        "What are your savings objectives?",
        "Source",
        "Reason_Equity",   
        "Reason_Mutual",   
        "Reason_Bonds",   
        "Reason_FD", "Invest_Monitor",     
    ]
    df.drop(columns=drop_cols, inplace=True)
    
    # Encode binary categorical (Label Encoding)
    le = LabelEncoder()
    df["gender"] = le.fit_transform(df["gender"])
    
    # # One-hot encode multi-class categoricals
    # ohe_cols = [
    #     "Factor", "Objective", "Purpose",
    #     "Invest_Monitor",
    #     "Reason_Equity", "Reason_Mutual", "Reason_Bonds", "Reason_FD"
    # ]
    # df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)
    
    # # 6. Convert bool columns to int
    # df = df.astype({col: int for col in df.select_dtypes(include="bool").columns})
    
    # Split features and target
    target = "Mutual_Funds"
    X = df.drop(columns=[target])
    y = df[target]
    
    # Train/test split FIRST (no resampling yet)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Resample ONLY the training set (to 500 rows)
    train_df = pd.concat([X_train, y_train], axis=1)
    train_resampled = resample(train_df, replace=True, n_samples=500, random_state=42)
    X_train_resampled = train_resampled.drop(columns=[target])
    y_train_resampled = train_resampled[target]
    
    # Scale features using training data only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_resampled)
    X_test_scaled = scaler.transform(X_test)
    
    # Convert back to DataFrame with column names (optional, for convenience)
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)
    
    return X_train_scaled, X_test_scaled, y_train_resampled, y_test, df

In [67]:
X_train, X_test, y_train, y_test = wrangle("/workspaces/Machine-Learining-Models/day_004_lasso_regression/finance/Finance_data.csv")

ValueError: could not convert string to float: 'Returns'

# MODEL and Evaluation

In [59]:

from sklearn.dummy import DummyRegressor
# Baseline
y_pred_mean = [y_train.mean()] * len(y_test)
baseline_mae = mean_absolute_error(y_test, y_pred_mean)
print(f"Baseline (mean prediction) Test MAE: {baseline_mae:.4f}")

# Lasso with cross-validation
lasso_cv = LassoCV(cv=5, random_state=42, alphas=np.logspace(-4, 1, 50))
lasso_cv.fit(X_train, y_train)

print(f"Best alpha: {lasso_cv.alpha_:.4f}")

y_pred = lasso_cv.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
lasso_mae = mean_absolute_error(y_test, y_pred)

print(f"Test MSE: {mse:.4f}")
print(f"Test R²: {r2:.4f}")
print(f"Lasso Test MAE: {lasso_mae:.4f}")

# 3. Feature selection
selected = X_train.columns[lasso_cv.coef_ != 0]
print(f"\nNumber of features selected: {len(selected)}")


ValueError: y_true and y_pred have different number of output (27!=1)